In [1]:
#import libraries
import numpy as np
from helper_ft import *#import helper functions
import helper_ft as hf
import list as lst
%load_ext autoreload
%aimport helper_ft
%autoreload 1

In [ ]:
def load_header(path):
    """Load column names from CSV header.
    path: path to CSV file
    return: list of column names in the header
    """
    with open(path, 'r') as f:
        header = f.readline().strip()
    return header.split(',')

def match_col_names(partial_names, header):
    """
    Match partial column names with full column names from the header.
    partial_names: list of partial column names to match
    header: list of all column names
    return: list of matched full column names
    """
    matched_names = []
    for pname in partial_names:
        for hname in header:
            if hname == pname:
                matched_names.append(hname)
                break
            elif hname.startswith(pname):
                matched_names.append(hname)
                break
            elif hname == f"_{pname}":
                matched_names.append(hname)
                break
    return matched_names

def get_col_index(cols, header):
    """Get the index of a column given its name from a list of column names.
    cols: list of column names to find
    header: list of all column names
    """
    #check name for name, if not found print an error message with the name not found and continue adding the found columns
    for name in cols:
        if name not in header:
            print(f"Column '{name}' not found in the provided list.")
    return [name for name in cols if name in header], [header.index(name) for name in cols if name in header]

def load_data_col(path, cols=(0), has_header=True):
    """Load numeric CSV with missing values -> np.nan using only numpy.
    path: path to CSV file
    cols: list of column indices to load (default: 0)
    has_header: whether the CSV file has a header row (default: True)
    """
    skip = 1 if has_header else 0
    data = np.genfromtxt(
        path,
        delimiter=",",
        skip_header=1 if has_header else 0,
        usecols=(cols),  # adjust based on relevant columns]       # force float to accommodate np.nan
        missing_values=["", "NA", "NaN"],       # treat empty strings as missing missing_values=["", "NA", "NaN"]
        filling_values=np.nan,
        autostrip=True,
        invalid_raise=False
    )
    return data

def expand_column(col, col_name):
    '''
    Expand a 1D column vector into a N array where N is the number of unique values in col.
    Each column in the output array is a binary indicator (0 or 1) of whether the corresponding
    entry in col matches the unique value for that column.
    col: 1D numpy array of categorical values

    #Missing values are handled by adding an additional binary indicator column and set to 0.
    '''
    is_missing = np.isnan(col).astype(np.int8)
    has_missing = np.any(is_missing)

    col = np.where(is_missing, 0, col)

    unique_values = np.unique(col)

    if unique_values.size > 10 and not has_missing:
        return None, None
    elif unique_values.size > 10 and has_missing:
        expanded = np.column_stack([col, is_missing])
        col_names = [f"{col_name}_is_missing"]
    else:
        #print(f"Unique values in column '{col_name}': {unique_values}")
        expanded = np.zeros((col.size, unique_values.size), dtype=np.int16)
        for i, val in enumerate(unique_values):
            expanded[:, i] = (col == val).astype(int)
        col_names = [f"{col_name}_{val}" for val in unique_values]

        if has_missing:
            expanded = np.column_stack([expanded, is_missing])
            col_names.append(f"{col_name}_is_missing")

    return expanded, col_names


def expand_dataset_col(col_list, path):
    '''
    Expand a dataset by selecting specific columns and expanding categorical columns into binary indicators.
    col_list: list of column names to select and expand
    header: list of all column names in the dataset
    dataset: numpy array of the dataset to expand or path to the dataset
    return: expanded dataset as a numpy array and list of new column names
    '''
    path_dataset = path
    header = load_header(path_dataset)
    print("Original col_list:", col_list)
    print("Header:", header)
    col_list = match_col_names(col_list, header)

    col_name, col_index = get_col_index(col_list, header)
    print(col_index)
    dataset = load_data_col(path_dataset, cols=col_index)
    header = load_header(path_dataset)
    x_train_subset = dataset
   
    col_list = match_col_names(col_list, header)
    print("Matched col_list:", col_list)
    col_list, col_index = get_col_index(col_list, header)
    print(len(col_index))
    #load x_train with only the columns in col_indices
    
    expanded_cols = []
    expanded_col_names = []
    for i, col_name in enumerate(col_list):
        expanded_col, col_names = expand_column(x_train_subset[:, i], col_name)
        if expanded_col is None:
            print(f"Skipping column '{col_name}' due to too many unique values.")
            continue
        expanded_cols.append(expanded_col)
        expanded_col_names.extend(col_names)
    return np.hstack(expanded_cols), expanded_col_names


In [47]:
dataset = load_data_col('../data/dataset/x_train.csv', cols=225)


In [49]:
dataset.shape

(328135,)

In [36]:
du = 0

In [37]:
np.any(du)

np.False_

In [50]:
dataset

array([nan, nan, nan, ..., nan, nan,  6.], shape=(328135,))

In [51]:
for x in range(20):
    print(f'x: {x}, dataset[x]: {dataset[x]}')

x: 0, dataset[x]: nan
x: 1, dataset[x]: nan
x: 2, dataset[x]: nan
x: 3, dataset[x]: nan
x: 4, dataset[x]: nan
x: 5, dataset[x]: nan
x: 6, dataset[x]: nan
x: 7, dataset[x]: nan
x: 8, dataset[x]: nan
x: 9, dataset[x]: nan
x: 10, dataset[x]: nan
x: 11, dataset[x]: nan
x: 12, dataset[x]: nan
x: 13, dataset[x]: nan
x: 14, dataset[x]: nan
x: 15, dataset[x]: 99.0
x: 16, dataset[x]: nan
x: 17, dataset[x]: nan
x: 18, dataset[x]: nan
x: 19, dataset[x]: 1.0


In [8]:
col_list = lst.col_list[:2]
col_list

['_RFHLTH', '_HCVU651']

In [52]:

x_train_expanded, expanded_col_names = expand_dataset_col(col_list, '../data/dataset/x_train.csv')


Original col_list: ['_RFHLTH', '_HCVU651']
Header: ['Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'COLGHOUS', 'STATERES', 'CELLFON3', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'CTELNUM1', 'CELLFON2', 'CADULT', 'PVTRESD2', 'CCLGHOUS', 'CSTATE', 'LANDLINE', 'HHADULT', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'CHOLCHK', 'TOLDHI2', 'CVDSTRK3', 'ASTHMA3', 'ASTHNOW', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABETE3', 'DIABAGE2', 'SEX', 'MARITAL', 'EDUCA', 'RENTHOM1', 'NUMHHOL2', 'NUMPHON2', 'CPDEMO1', 'VETERAN3', 'EMPLOY1', 'CHILDREN', 'INCOME2', 'INTERNET', 'WEIGHT2', 'HEIGHT3', 'PREGNANT', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'SMOKDAY2', 'STOPSMK2', 'LASTSMK2', 'USENOW3', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'MAXDRNKS', 'FRUITJU1', 'FRUIT1', 'FVBEANS',

In [10]:
import pandas as pd

<function __main__.expand_dataset_col(col_list, path)>

In [60]:
pd_data = pd.DataFrame(x_train_expanded, columns=expanded_col_names)
pd_data.head(20)

,_RFHLTH_1.0,_RFHLTH_2.0,_RFHLTH_9.0,_RFHLTH_is_missing,_HCVU651_1.0,_HCVU651_2.0,_HCVU651_9.0,_HCVU651_is_missing
0,1,0,0,0,1,0,0,0
1,0,1,0,0,1,0,0,0
2,1,0,0,0,0,0,1,0
3,1,0,0,0,0,0,1,0
4,1,0,0,0,0,0,1,0
5,1,0,0,0,1,0,0,0
6,0,1,0,0,0,0,1,0
7,1,0,0,0,0,0,1,0
8,1,0,0,0,1,0,0,0
9,1,0,0,0,1,0,0,0


In [61]:
pd_data.columns

Index(['_RFHLTH_1.0', '_RFHLTH_2.0', '_RFHLTH_9.0', '_RFHLTH_is_missing',
       '_HCVU651_1.0', '_HCVU651_2.0', '_HCVU651_9.0', '_HCVU651_is_missing'],
      dtype='object')

In [12]:
expanded_col_names

['_RFHLTH_1.0',
 '_RFHLTH_2.0',
 '_RFHLTH_9.0',
 '_HCVU651_1.0',
 '_HCVU651_2.0',
 '_HCVU651_9.0']

In [13]:
x_train_expanded.shape

(328135, 6)